##DATA EXPLORATION

In [ ]:
#imports
import string

import pandas as pd   #handles data tables (for data manipulation and analysis)
import numpy as np  #handles vectors
import matplotlib.pyplot as plt   #for graphs
import seaborn as sns

import nltk
from nltk.tokenize import word_tokenize  #splits text into words
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#ML tools
from sklearn.model_selection import train_test_split  #Model selection
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer   #Feature extraction
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report  #Evaluation metrics
from sklearn.tree import DecisionTreeClassifier

#Models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
!pip install gensim
from gensim.models import Word2Vec  #builds embeddings

#NLTK downloads - needed for preprocessing
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
#Connect to google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#data handling

# Read and load the two news CSV datasets into table(dataframe)
df_fake = pd.read_csv('/content/drive/MyDrive/COMP316 Project/data/Fake.csv')
df_real = pd.read_csv('/content/drive/MyDrive/COMP316 Project/data/True.csv')

# Labels: 0 - fake; 1- real
df_fake['label'] = 0   # fake news
df_real['label'] = 1   # real news

# Merge datasets (fake and real)
data = pd.concat([df_fake, df_real])

# Resets index so rows are unique
data = data.reset_index(drop=True)

# Keep only necessary columns (to keep model clean) - removes fields like date, subject...
data = data[['text', 'label']]

# view data
data.head()

# check structure from file
data.info()

# check class distribution (balanced/unbalanced)
data['label'].value_counts()

# plot graph
data['label'].value_counts().plot(kind='bar')
plt.title("Fake (0) vs Real (1) Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()

# text length analysis - checking distribution to understand dataset characteristics and verify preprocessing didn't truncate content
data['text_length'] = data['text'].apply(len)

data['text_length'].hist()
plt.title("Text Length Distribution")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.show()


##METHODS SUPPORTING MODULAR DESIGN

In [ ]:
#returns vectorizer name for experiment
def get_vectorizer(name):

    if name == "count_uni":
        return CountVectorizer(ngram_range=(1,1))

    elif name == "count_bi":
        return CountVectorizer(ngram_range=(1,2))

    elif name == "tfidf_uni":
        return TfidfVectorizer(ngram_range=(1,1))

    elif name == "tfidf_bi":
        return TfidfVectorizer(ngram_range=(1,2))

    elif name == "word2vec":
      return "word2vec"

    else:
        raise ValueError("Invalid vectorizer choice")

In [ ]:
#returns ML model we are training
def get_model(name):

    if name == "nb":
        return MultinomialNB()

    elif name == "lr":
        return LogisticRegression(max_iter=1000)

    elif name == "svm":
        return SVC()

    elif name == "dt":
        return DecisionTreeClassifier()

    elif name == "lsvc":
        return LinearSVC(max_iter=1000)

    else:
        raise ValueError("Invalid model choice")

In [ ]:
# performs series of tasks to follow standard pipeline, based off modular design
def run_pipeline(vectorizer_name, model_name, X_train, X_test, y_train, y_test):

  if vectorizer_name == "word2vec":
    X_train_tokens = X_train.apply(word_tokenize)
    X_test_tokens = X_test.apply(word_tokenize)

    w2v_model = Word2Vec(
        sentences=X_train_tokens,
        vector_size=100,
        window=5,
        min_count=2
    )

    X_train_vec = np.array([get_avg_word2vec(t, w2v_model, 100) for t in X_train_tokens])
    X_test_vec = np.array([get_avg_word2vec(t, w2v_model, 100) for t in X_test_tokens])

  else:
    # 1.Choose vectorizer
    vectorizer = get_vectorizer(vectorizer_name)

    # 2.Convert text → numbers
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

  # 3.Choose model
  model = get_model(model_name)

  # 4.Train model
  model.fit(X_train_vec, y_train)

  # 5.Predict
  predictions = model.predict(X_test_vec)

  # 6.Evaluate
  acc = accuracy_score(y_test, predictions)
  prec = precision_score(y_test, predictions)
  rec = recall_score(y_test, predictions)
  f1 = f1_score(y_test, predictions)

  # 7.Print results
  print("Vectorizer:", vectorizer_name)
  print("Model:", model_name)
  print("Accuracy:", acc)
  print("Precision:", prec)
  print("Recall:", rec)
  print("F1 Score:", f1)

  print( classification_report(y_test, predictions)) #precision, recall per class- useful for reports

  cm = confusion_matrix(y_test, predictions)

  plt.figure()
  sns.heatmap(cm, annot=True, fmt='d')
  plt.title(f"Confusion Matrix ({vectorizer_name} + {model_name})")
  plt.xlabel("Predicted")
  plt.ylabel("Actual")
  plt.savefig(f"{vectorizer_name}_{model_name}.png")  #saves graph for final report
  plt.show()

  return acc, prec, rec, f1


In [ ]:
#get average length of sentence to perform word2vec
def get_avg_word2vec(tokens, model, vector_size):

    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)


##DATA PREPROCESSING (CLEANING TEXT FROM DATASET)

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    if not isinstance(text, str):
      return ""

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    no_punct = ""
    for char in text:
        if char not in string.punctuation: # string.punctuation has .!,?:; etc punctuation marks
            no_punct = no_punct + char

    # Tokenize(split into words)
    words = word_tokenize(no_punct)

    # Remove stopwords
    filtered_words = []
    for w in words:
        if w not in stop_words:
            filtered_words.append(w)

    # Lemmatization
    lemmatized_words = []
    for w in filtered_words:
        lemma = lemmatizer.lemmatize(w)
        lemmatized_words.append(lemma)

    # Join words back into a sentence
    clean_sentence = ""
    for w in lemmatized_words:
        clean_sentence = clean_sentence + w + " "

    return clean_sentence.strip()

# Clean the data
clean_texts = []

for text in data['text']:
    cleaned = clean_text(text)
    clean_texts.append(cleaned)

data['clean_text'] = clean_texts

# Display some of the data vs clean(preprocessed) data
for i in range(5):
    print("ORIGINAL:")
    print(data['text'][i])
    print("CLEANED:")
    print(data['clean_text'][i])
    print("------")

rows_to_keep = []

for i in range(len(data)):
    text = data['clean_text'][i]

    if text.strip() != "":
        rows_to_keep.append(i)

data = data.loc[rows_to_keep]
data = data.reset_index(drop=True)

# COUNT LABELS
fake_count = 0
real_count = 0

for label in data['label']:
    if label == 0:
        fake_count = fake_count + 1
    else:
        real_count = real_count + 1

print("Fake news:", fake_count)
print("Real news:", real_count)

# Save cleaned dataset to a NEW CSV file
output_path = "/content/drive/MyDrive/COMP316 Project/data/clean_fake_news.csv"

data.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")

##DATA SPLIT AND PREPARATION

In [ ]:
#split data into input/output
X_text = data['clean_text']
y = data['label']

In [ ]:
#train/test split (We split ONCE - so all experiments are fair)
X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42)

##EXPERIMENT SECTION

In [ ]:
# --- INTERACTIVE EXPERIMENT LOOP ---
#systematic experimentation- fair comparison, strong methodology  then save results in table

results = []

while True:
    print("\nChoose vectorizer:")
    print("1. count_uni")
    print("2. count_bi")
    print("3. tfidf_uni")
    print("4. tfidf_bi")
    print("5. word2vec")
    print("6. stop")

    choice = input("Enter choice: ").strip()

    if choice == "6":
        break

    # map user input
    vectorizer_map = {
        "1": "count_uni",
        "2": "count_bi",
        "3": "tfidf_uni",
        "4": "tfidf_bi",
        "5": "word2vec"
    }

    vectorizer = vectorizer_map.get(choice)

    if vectorizer is None:
        print("Invalid choice")
        continue

    # choose models
    if vectorizer == "word2vec":
        models = ["lr", "svm", "lsvc", "dt"] # Naive Bayes excluded because Word2Vec embeddings may contain negative values
    else:
        models = ["nb", "lr", "svm", "lsvc", "dt"]

    print(f"\nRunning all models for: {vectorizer}")

    for model in models:
        print("Running:", vectorizer, "+", model)

        acc, prec, rec, f1 = run_pipeline(
            vectorizer, model,
            X_train, X_test,
            y_train, y_test
        )

        results.append([vectorizer, model, acc, prec, rec, f1])

    print("\nFinished this batch!")

# --- FINAL RESULTS ---
results_df = pd.DataFrame(results, columns=["Vectorizer", "Model", "Accuracy", "Precision", "Recall", "F1"])

print("\nFINAL RESULTS:")
print(results_df)

results_df.to_csv('/content/drive/MyDrive/COMP316 Project/results/final_results.csv', index=False) #so we don't lose anything

#display sorted results in bar graph (using F1 metric) to compare results visually
sorted_results = results_df.sort_values(by="F1", ascending=False)

plt.figure(figsize=(12, 6))
names = sorted_results["Vectorizer"] + " + " + sorted_results["Model"]
scores = sorted_results["F1"]

plt.bar(names, scores, color='steelblue', edgecolor='white')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title("Model Comparison (Sorted by F1 Score)", fontsize=13)
plt.ylabel("F1 Score")
plt.ylim(0.92, 1.0)
plt.xlabel("Vectorizer + Model Configuration")
plt.tight_layout()
plt.show()